# 01 — The artificial neuron == the logistic unit you already built

Welcome to the first method beyond trees — and a gentle place to start, because you have **already
built its building block**. This module introduces the **multi-layer perceptron (MLP)**, a model
that learns its own features. Before we stack anything, we meet that block — the **artificial
neuron** — and find it is an old friend wearing a new name.

**Prerequisites**
- Chapter 03 — NB 1 (the sigmoid σ), NB 3 (the log-loss), NB 4 (gradient descent).
- Chapter 00 — the train/test split and accuracy.

**What you'll be able to do**
- Describe a neuron as a weighted sum, a bias, and an activation.
- Explain why a single sigmoid neuron *is* logistic regression.
- Name three activation functions — sigmoid, tanh, ReLU — and say what each does.
- Show that an MLP with no hidden layer reproduces logistic regression.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import expit  # the sigmoid, computed stably
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

from ml_course import colors, viz

viz.use_course_style()

SEED = 0
np.random.seed(SEED)

## A quick recap: what chapter 03 already built

In chapter 03 we built a binary classifier by hand, one piece at a time:

- a **linear score** `z = w·x + b` — each feature weighted, plus a bias;
- the **sigmoid** `σ(z) = 1 / (1 + e^(−z))`, which squashes that score into a probability in (0, 1);
- the **log-loss**, the objective we minimized;
- **gradient descent**, the loop that found the weights `(w, b)`.

This notebook leans on all four. We are not replacing them — we are about to look at the same machine
from a new angle.

## A neuron, in three steps

An **artificial neuron** does three things, in order:

1. **Weighted sum** — multiply each input by a weight and add them up: `w·x`.
2. **Bias** — add a constant offset `b`, giving the score `z = w·x + b`.
3. **Activation** — pass `z` through a function `φ` to get the output: `a = φ(z)`.

The first two steps are exactly the linear score from chapter 03. The third — the **activation φ** —
is the new word, and it is a *pluggable* choice: swap φ and you change what the neuron computes.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.axis("off")
C = colors.COLORS

# Two input nodes on the left.
inputs = [(0.0, 1.4, "x₁"), (0.0, 0.2, "x₂")]
for ix, iy, label in inputs:
    ax.add_patch(plt.Circle((ix, iy), 0.18, color=C["class_a"], ec=C["text"], lw=1.2, zorder=3))
    ax.text(ix, iy, label, ha="center", va="center", color=C["text"], zorder=4)

# The weighted-sum-plus-bias node, then the activation node.
sumx, sumy = 1.7, 0.8
phix, phiy = 3.1, 0.8
ax.add_patch(plt.Circle((sumx, sumy), 0.27, color=C["class_c"], ec=C["text"], lw=1.2, zorder=3))
ax.text(sumx, sumy, "Σ + b", ha="center", va="center", color=C["text"], zorder=4)
ax.add_patch(plt.Circle((phix, phiy), 0.27, color=C["class_b"], ec=C["text"], lw=1.2, zorder=3))
ax.text(phix, phiy, "φ", ha="center", va="center", color=C["text"], fontsize=15, zorder=4)
ax.text(4.7, 0.8, "a = φ(w·x + b)", ha="left", va="center", color=C["text"], fontsize=13)

# Arrows: inputs -> sum (with weight labels), sum -> activation, activation -> output.
arrow = dict(arrowstyle="->", color=C["muted"], lw=1.4)
for (ix, iy, _), wlabel in zip(inputs, ["w₁", "w₂"], strict=True):
    ax.annotate("", xy=(sumx - 0.27, sumy), xytext=(ix + 0.18, iy), arrowprops=arrow)
    ax.text((ix + sumx) / 2, (iy + sumy) / 2 + 0.1, wlabel, ha="center", color=C["text"])
ax.annotate("", xy=(phix - 0.27, phiy), xytext=(sumx + 0.27, sumy), arrowprops=arrow)
ax.annotate("", xy=(4.6, 0.8), xytext=(phix + 0.27, phiy), arrowprops=arrow)

ax.set_xlim(-0.4, 6.6)
ax.set_ylim(-0.2, 1.9)
ax.set_title("One artificial neuron")
plt.show()

**Read the figure.** Two inputs `x₁` and `x₂` enter on the left. Each is multiplied by its weight
(`w₁`, `w₂`) along the arrows, summed together with the bias `b` in the middle node, and the
resulting score is passed through the activation `φ` on the right. The single number that comes out,
`a = φ(w·x + b)`, is the neuron's output. One neuron, three steps.

## Meet three activations

The activation φ is where we get to choose. Three are worth knowing from the start:

- **Sigmoid** `σ(z) = 1/(1+e^(−z))` — smooth, squashes to (0, 1). The one chapter 03 used to read a
  score as a probability.
- **tanh** `tanh(z)` — sigmoid's close cousin, smooth and squashing to (−1, 1), centred at zero.
- **ReLU** `max(0, z)` — off (zero) below zero, then linear. The modern default, for reasons we
  meet later.

*Why* we need a non-linear φ at all — rather than reading `z` directly — is the question of the next
notebook. Here we meet the candidates.

In [ ]:
z = np.linspace(-5, 5, 400)
curves = {
    "sigmoid  σ(z)": (expit(z), colors.COLORS["class_a"]),
    "tanh(z)": (np.tanh(z), colors.COLORS["class_b"]),
    "ReLU  max(0, z)": (np.maximum(0.0, z), colors.COLORS["class_c"]),
}

fig, ax = plt.subplots(figsize=(7.5, 5))
for label, (values, color) in curves.items():
    ax.plot(z, values, color=color, lw=2.4, label=label)
ax.axhline(0, color=colors.COLORS["grid"], lw=1)
ax.axvline(0, color=colors.COLORS["grid"], lw=1)
ax.set_xlabel("z   (the score w·x + b)")
ax.set_ylabel("φ(z)   (the neuron's output)")
ax.set_title("Three activation functions")
ax.legend()
plt.show()

**Read the figure.** All three take the same score `z` on the x-axis and reshape it. The **sigmoid**
rises smoothly from 0 to 1 — its output reads as a probability. **tanh** has the same S-shape but
ranges from −1 to +1 and is centred at zero. **ReLU** is the simplest non-linearity of the three: two
straight pieces meeting at a single kink at zero — flat for negative scores, then rising. Same score
in, three different outputs. We use the smooth **sigmoid** for the rest of this notebook; **tanh** and
**ReLU** are the activations the *hidden* neurons of the next notebooks will lean on.

## The bridge: a sigmoid neuron *is* logistic regression

Look again at chapter 03. Logistic regression predicts `p = σ(w·x + b)`. That is **exactly** a single
neuron with a sigmoid activation: a weighted sum, a bias, and σ. Nothing has been added. The "neuron"
is not a new model — it is logistic regression, renamed.

Let us prove it two ways: first by hand, then with scikit-learn's own MLP.

In [ ]:
# Fit logistic regression on a small 2-D dataset.
X, y = make_moons(n_samples=400, noise=0.20, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)

logreg = LogisticRegression().fit(X_train, y_train)

# Read off its learned weights and bias.
w = logreg.coef_.ravel()
b = logreg.intercept_[0]
print("weights w =", w)
print("bias    b =", float(b))

# Compute the sigmoid neuron BY HAND: a = σ(X w + b).
proba_by_hand = expit(X_test @ w + b)

# scikit-learn's own probability for the positive class.
proba_sklearn = logreg.predict_proba(X_test)[:, 1]

max_diff = np.max(np.abs(proba_by_hand - proba_sklearn))
print("max | by-hand sigmoid neuron − logistic.predict_proba | =", f"{max_diff:.2e}")

**Read the result.** The largest difference between our by-hand sigmoid neuron and scikit-learn's
`predict_proba` is `0.00e+00` — they agree to the last bit. This is not an approximation: a sigmoid
neuron and logistic regression compute the identical function. Same weights, same bias, same σ, same
probabilities.

## scikit-learn's MLP, with no hidden layer

`MLPClassifier` is the library's multi-layer perceptron. Give it `hidden_layer_sizes=()` — an empty
tuple, *no hidden layer* — and all that remains is the output neuron. With a sigmoid activation, that
output neuron behaves like logistic regression. Let's check on the same data.

In [ ]:
mlp_empty = MLPClassifier(
    hidden_layer_sizes=(),    # no hidden layer: only the output neuron remains
    activation="logistic",    # the sigmoid
    solver="lbfgs",           # a full-batch optimizer, like the one chapter 03 fit with
    alpha=1e-4,
    max_iter=5000,
    random_state=SEED,
).fit(X_train, y_train)

acc_logreg = logreg.score(X_test, y_test)
acc_mlp = mlp_empty.score(X_test, y_test)
print(f"logistic regression   test accuracy = {acc_logreg:.4f}")
print(f"MLP, no hidden layer   test accuracy = {acc_mlp:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
viz.plot_decision_boundary(logreg, X_test, y_test, ax=axes[0])
axes[0].set_title(f"Logistic regression  (acc {acc_logreg:.3f})")
viz.plot_decision_boundary(mlp_empty, X_test, y_test, ax=axes[1])
axes[1].set_title(f"MLP, no hidden layer  (acc {acc_mlp:.3f})")
plt.show()

**Read the figure.** Two panels, the same straight boundary, the same accuracy. The empty-hidden MLP
and logistic regression drew the identical line — because they *are* the same model: a single sigmoid
neuron. Their raw weights differ a little, but not because of the optimizer (both use `lbfgs` here) —
it is the **default L2 penalty**: `LogisticRegression`'s `C=1` shrinks the weights more than the MLP's
`alpha=1e-4`. Match the regularization and the weight vectors coincide; the boundary and accuracy are
the same either way. Notice, too, that a straight line leaves some moon-shaped points on the wrong
side — a hint of what one neuron **cannot** do. That is exactly where the next notebook begins.

## Your turn

1. **(warm-up)** In the empty-hidden `MLPClassifier`, change `activation="logistic"` to
   `activation="tanh"`. Re-fit and re-plot. Does the boundary stay straight? Why would changing the
   activation of a *single* output neuron not bend the boundary?
2. **(core)** Increase the moons `noise` to 0.35 and re-run the comparison. Do logistic regression
   and the empty-hidden MLP still reach nearly the same test accuracy? They should track each other —
   they are the same model.
3. **(reach)** tanh and sigmoid are relatives. Verify numerically that `σ(z) = (tanh(z/2) + 1) / 2`
   on a grid of `z` values, then explain in one sentence why a tanh "neuron" can represent the same
   boundary as a sigmoid one.

## What you built

- You reframed the linear score `w·x + b` and the sigmoid as a single **artificial neuron**:
  weighted sum → bias → activation.
- You met three activations — **sigmoid**, **tanh**, **ReLU** — and saw how each reshapes the score.
- You proved, to the last bit, that a **sigmoid neuron is logistic regression** (max difference
  `0.00e+00`).
- You watched scikit-learn's `MLPClassifier(hidden_layer_sizes=())` reproduce logistic regression's
  boundary and accuracy.

The neuron is a friend you already knew. Next, we give it company.

## Going further (optional)

The very first artificial neuron — Rosenblatt's **perceptron** (1958) — used a hard **step**
activation: output 1 if the score crossed a threshold, 0 otherwise. It worked, but a step has no
usable slope, so it could not be trained by gradient descent. The **sigmoid** neuron is its smooth,
differentiable cousin: because σ has a gradient everywhere, the descent loop from chapter 03 can tune
its weights. That single change — a smooth activation — is what later made stacking neurons into
trainable networks possible.

## References

- Rosenblatt, F. (1958). *The perceptron: a probabilistic model for information storage and
  organization in the brain.* Psychological Review 65(6), 386–408.
  https://doi.org/10.1037/h0042519
- Chapter 03 — NB 1 (the sigmoid), NB 3 (the log-loss), NB 4 (gradient descent).

*Previous:* chapter 10 — LightGBM.  *Next:* **11.2 — why one neuron is not enough (the hidden layer).**